# Data-Science Notebook on SPUR (DuckDB) — design + tech spec

> **Status:** design approved (2026-05-29). Authoritative spec. Brainstorm trail + interactive exploration live in `docs/rca/2026-05-29-data-science-notebook-duckdb-proposal.ipynb`.
> **Vision:** attach a dataset/datasource to a Jute notebook (GUI sidebar drag-drop *or* a TUI slash command), then drive analysis + visualization by asking the SPUR brain agent from spur-tui. **DuckDB is the data-processing engine.**
> Diagrams are normative; prose is supporting.

## 1. Decisions (locked)

| Axis | Decision |
|---|---|
| Engine | **DuckDB**, executed **inside the Python kernel** (`duckdb` pkg) |
| MVP scope | **A** local files (CSV / Parquet / JSON); B/C/D via extensions later |
| Representation | **C** — `AttachDatasource` command → daemon inserts reproducible **setup cell** + records a **schema catalog** |
| Brain awareness | on-demand **`notebook_list_datasources`** MCP tool |
| Attach entrypoints | **GUI sidebar** (drag-drop / Add) **and** TUI `/notebook data add` |
| Multiplicity | many datasources, optional **grouping** |
| TUI reference | **`@`-mention** a datasource (new mention source) |
| Visualization | reuse existing kernel + `OutputView` iframe (no new viz code) |

## 2. Scope & engine — one extension-driven arc

Choosing DuckDB makes scope expansion **“enable an extension,”** not “build a connector.” The same SQL surface that reads a local CSV today attaches Postgres or S3 later.

| Scope | Data reached | DuckDB mechanism | New plumbing |
|---|---|---|---|
| **A** (MVP) | CSV / Parquet / JSON (local) | `read_csv_auto` / `read_parquet` / `read_json_auto` | none |
| **B** | DuckDB / SQLite files, **spur-analyst graph index** | `ATTACH`, `sqlite_scanner` | minimal |
| **C** (later) | Postgres / MySQL | `postgres_scanner` / `mysql_scanner` | secrets mgmt |
| **D** (later) | S3 / https / Iceberg / Delta | `httpfs` / `iceberg` / `delta` | credentials |

**Engine strategy (corrected):** DuckDB is becoming a **hard, shared workspace dependency** (spur-analyst MCP, cost, context engine, …); the notebook daemon **reuses** it rather than adding a bespoke engine. (Today DuckDB is an optional `spur-context` feature and `spur-analyst` uses the DuckDB CLI in tests — it is *not* yet embedded in `spur-analyst`; the §7 daemon probe is the first embed, justified as shared-engine reuse.) **Visualization already works** (kernel `display_data` → `OutputView` iframe renders `image/png`). The genuinely new surface is **attach + brain-awareness**, not run/render.

## 3. End-to-end journey

```mermaid
sequenceDiagram
  autonumber
  participant U as User
  participant SB as Jute sidebar (GUI)
  participant D as daemon (+catalog)
  participant K as Python kernel (duckdb)
  participant B as Brain (from spur-tui)
  participant C as Notebook cells

  U->>SB: drag sales.csv (or /notebook data add)
  SB->>D: AttachDatasource{path, name:'sales'}
  D->>D: introspect schema + insert setup cell + update catalog
  D-->>SB: sidebar shows 'sales' + columns
  Note over U,B: user switches to spur-tui
  U->>B: '@sales plot monthly revenue by region'
  B->>D: notebook_list_datasources (schema-aware)
  B->>D: insert + run duckdb.sql(...).df().plot()
  K-->>C: display_data image/png (chart)
  Note over U,C: chart renders live in Jute
```

## 4. Component architecture (layer-level)

New in **green**, touched-existing in **amber**, reused-as-is in grey.

```mermaid
flowchart TB
  subgraph FE[jute-notebook frontend - React]
    SB[DatasourceSidebar.tsx]:::new
    CTRL[daemon/control.ts attach helpers]:::edit
    OV[OutputView.tsx - charts]:::keep
  end
  subgraph JUTE[jute lib - src-tauri]
    CMD[DaemonControlCommand::AttachDatasource]:::new
    CAT[DatasourceCatalog + DatasourceEntry]:::new
    INTRO[duckdb introspect - schema only]:::new
    SETUP[setup-cell generator]:::new
    STORE[NotebookStore]:::keep
  end
  subgraph BIN[spur-notebook bin]
    ROUTER[mcp/mod.rs handle router]:::edit
    TOOL[notebook_list_datasources MCP tool]:::new
    EVT[DatasourcesChanged emit]:::new
  end
  subgraph TUI[spur-tui]
    SLASH[/notebook data add - slash_commands.rs/]:::edit
    DSRC[DatasourceMentionSource]:::new
    KIND[MentionKind::Datasource - exhaustive fan-out]:::edit
    EVH[events.rs - DatasourcesChanged handler]:::edit
  end
  subgraph K[Python kernel]
    DUCKK[duckdb analysis connection]:::keep
  end
  BRAIN[Brain agent - ACP]:::keep

  SB --> CTRL --> CMD
  SLASH --> CMD
  BRAIN -- notebook_* MCP --> ROUTER
  CMD --> ROUTER
  ROUTER --> CAT
  ROUTER --> INTRO
  ROUTER --> SETUP --> STORE
  CAT --> TOOL --> BRAIN
  CAT --> EVT --> EVH --> DSRC --> KIND
  STORE --> DUCKK --> OV
  classDef new fill:#dcfce7,stroke:#16a34a,color:#14532d;
  classDef edit fill:#fef9c3,stroke:#ca8a04,color:#713f12;
  classDef keep fill:#f1f5f9,stroke:#94a3b8,color:#334155;
```

## 5. Data model

> **Decision (user, overriding codex's polling fallback):** **event-driven / push**, not polling. The daemon gains its **own in-process broadcast `event_tx`**; the orchestrator bridges it onto the **existing `SpurEvent` bus** the TUI already consumes (same delivery path as `NotebookSocketReady`). See §16 for the channel. The catalog still **persists in notebook metadata** so it survives reload/restart.

```mermaid
classDiagram
  class DatasourceEntry {
    +String name
    +PathBuf path
    +DatasourceKind kind
    +Option~String~ group
    +Vec~Column~ columns
    +Option~u64~ row_count
  }
  class Column {
    +String name
    +String sql_type
  }
  class DatasourceKind {
    <<enum>>
    Csv
    Parquet
    Json
  }
  class DatasourceCatalog {
    +u32 schema_version
    +Vec~DatasourceEntry~ entries
    +event_tx broadcast~DaemonEvent~
    +attach(entry)
    +detach(name)
    +list() Vec~DatasourceEntry~
    +hydrate_from_metadata(meta)
    +persist_to_metadata() meta
  }
  class AttachDatasource {
    <<DaemonControlCommand>>
    +String name
    +PathBuf path
    +Option~String~ group
  }
  class DaemonEvent {
    <<in-daemon broadcast>>
    DatasourcesChanged(Vec~DatasourceEntry~)
  }
  class DatasourcesChanged {
    <<SpurEventBody>>
    +SessionId session
    +Vec~DatasourceEntry~ entries
  }
  DatasourceCatalog o-- DatasourceEntry
  DatasourceEntry *-- Column
  DatasourceEntry --> DatasourceKind
  AttachDatasource ..> DatasourceEntry : produces
  DatasourceCatalog ..> DaemonEvent : emits on change
  DaemonEvent ..> DatasourcesChanged : bridged by orchestrator
```

**Catalog persistence contract:** the `DatasourceCatalog` lives in-memory on Jute `State`. Entries serialize into **notebook metadata** (`metadata.spur.datasources`, with `schema_version`) on change, and **hydrate on notebook load**. Paths are normalized (absolute, workspace-relative where possible) so a re-opened notebook re-resolves its sources.

## 6. Attach flow (both entrypoints → one command → push)

> **Event-driven:** the daemon emits on its internal `event_tx`; the orchestrator's persistent subscriber bridges it onto the `SpurEvent` bus; the TUI updates its mention snapshot from the event it already receives in `events.rs`. No polling.

```mermaid
sequenceDiagram
  autonumber
  participant SB as GUI sidebar
  participant TUI as spur-tui (/notebook data add)
  participant R as daemon router (mcp/mod.rs)
  participant I as duckdb introspect (Rust, schema-only)
  participant CAT as DatasourceCatalog (+event_tx)
  participant ST as NotebookStore
  participant BR as orchestrator bridge (subscribe loop)
  participant EV as SpurEvent bus
  participant M as TUI MentionRegistry

  alt GUI drag-drop
    SB->>R: AttachDatasource{name,path,group}
  else TUI slash
    TUI->>R: AttachDatasource{...} (DaemonControlRequest)
  end
  R->>I: DESCRIBE / SUMMARIZE read_csv_auto(path)
  I-->>R: columns[], row_count
  R->>CAT: upsert DatasourceEntry
  R->>ST: insert/UPDATE idempotent setup cell (CREATE VIEW)
  Note over CAT,ST: persist catalog into notebook metadata
  CAT-->>BR: DaemonEvent::DatasourcesChanged (push frame over daemon socket)
  BR->>EV: emit SpurEvent::DatasourcesChanged{session, entries}
  EV-->>M: handle_spur_event -> set_datasource_snapshot(entries)
  Note over M: @-mention 'Data' section updates proactively
  R-->>SB: ok (sidebar shows schema)
```

## 7. DuckDB engine placement (resolved: shared workspace engine)

> **Review outcome (codex `d1b22682`):** keep **two DuckDB execution contexts**, on a *corrected* rationale. The earlier "spur-analyst embeds DuckDB in Rust" precedent was wrong. The real justification is forward-looking: DuckDB is becoming a **hard, shared workspace dependency** (spur-analyst MCP, cost, context engine, …), so a daemon-side DuckDB is **reuse of the platform engine, not a redundant new dependency.**

**Two contexts, one shared engine + version:**
- **Daemon-side DuckDB (Rust)** — read-only **schema probe** at attach (`DESCRIBE`): populates the catalog (columns; `row_count` best-effort) **before any kernel exists**. This is what makes attach-time catalog + `@`-mentions work without booting a kernel.
- **Kernel-side DuckDB (Python)** — the **only** analysis + render runtime (`con.sql(...).df().plot()` → `display_data` → OutputView).

**Why not collapse to one:**
- *kernel-only*: cannot satisfy attach-time schema without auto-starting a hidden kernel; couples `@`-mention availability to kernel boot + the `duckdb` pip pkg.
- *daemon-only*: would reinvent the viz/result bridge — violates the locked "viz in the kernel" model.

```mermaid
flowchart LR
  subgraph SHARED[shared workspace DuckDB - one version]
    note[consumers: spur-analyst MCP, cost, context engine, notebook daemon]
  end
  subgraph daemon[notebook daemon - Rust]
    D1[schema probe - read-only DESCRIBE]:::a
  end
  subgraph kernel[Python kernel]
    D2[analysis + viz - duckdb pip]:::b
  end
  SHARED -. same engine/version .-> D1
  FILE[(sales.csv)] --> D1 --> CAT[catalog]
  FILE --> D2 --> OUT[display_data -> OutputView]
  classDef a fill:#e0f2fe,stroke:#0284c7,color:#075985;
  classDef b fill:#ecfccb,stroke:#65a30d,color:#365314;
```

**Dependency & skew contract:** one workspace DuckDB version shared by all Rust consumers; the kernel `duckdb` pip version tracks it. **Catalog schema is advisory** (the kernel is authoritative) — refresh/warn on mismatch. Do **not** build a daemon-side query/analysis engine; the daemon DuckDB is schema-probe-only.

> **Grounded correction (codex `e38ffef2`):** "shared workspace dependency" is only *partly* true today — the DuckDB **version** is in workspace deps, but the actual consumers are **feature-gated (`spur-context`) or CLI-based (spur-analyst tests)**, and the **notebook crate has no DuckDB dependency at all yet.** So this is a real **new dependency decision for `jute`/`spur-notebook`**, not free reuse.
>
> **Decision required before T-introspection (see §11):** add a DuckDB dependency to the daemon crate behind a **feature gate** (e.g. `datasource-introspect`), pinned to the workspace version, and explicitly **accept the compile-cost / binary-size delta**. If that cost is rejected, the fallback is to defer schema probing to a short-lived kernel call (couples `@`-mentions to kernel boot — the daemon-only/kernel-only tradeoffs above).

## 8. Analysis + visualization flow

```mermaid
sequenceDiagram
  autonumber
  participant U as User (spur-tui)
  participant M as Mention picker
  participant B as Brain (ACP)
  participant T as notebook_list_datasources (MCP)
  participant ST as NotebookStore
  participant K as kernel (duckdb)
  participant OV as OutputView (Jute)

  U->>M: types '@' -> Data section -> @sales
  M-->>U: expands to view-ref + schema hint
  U->>B: 'plot monthly revenue by region using @sales'
  B->>T: notebook_list_datasources()
  T-->>B: [sales: region,month,revenue,...]
  B->>ST: notebook_insert_cell(duckdb.sql(...).df().plot())
  B->>K: notebook_run_cell
  K-->>OV: display_data image/png
  OV-->>U: chart renders in Jute
```

## 9. Datasource lifecycle

```mermaid
stateDiagram-v2
  [*] --> Attaching: AttachDatasource
  Attaching --> Introspecting: file readable
  Attaching --> Error: file missing / unreadable
  Introspecting --> Ready: schema captured + setup cell inserted + event emitted
  Introspecting --> Error: DuckDB parse failure
  Ready --> Ready: re-run notebook (setup cell idempotent)
  Ready --> Detached: DetachDatasource
  Error --> Attaching: retry
  Detached --> [*]
```

## 10. TUI mentions integration + the fan-out

`@`-mention reuses the existing `MentionRegistry`/`MentionSource` framework. The snapshot is refreshed **proactively** from the bridged `SpurEvent::DatasourcesChanged` (handled in `events.rs` next to `NotebookSocketReady`). The cost is the new **`MentionKind::Datasource`** variant, matched **exhaustively** in `registry.rs` (`code_match_rank`, `code_entry_path`, `empty_code_kind_rank`, `tier_rank`, + empty-query sectioning) — mechanical but must be exhaustive, so it is its own isolated task.

> **Grounded (codex `e38ffef2`):** a datasource mention today would only resolve to a generic `ResourceLink`. Define a **`datasource://<name>` URI** and a **prompt-hint expansion** so the brain receives the source's schema/columns (not just an opaque link) when a `@datasource` is mentioned.

```mermaid
flowchart TB
  EVT[SpurEvent::DatasourcesChanged - pushed via bridge]:::new --> SNAP[MentionRegistry::set_datasource_snapshot]:::new
  SNAP --> SRC[DatasourceMentionSource::build]:::new
  SRC --> SEC[empty-query 'Data' section + DATASOURCE_CAPk]:::edit
  SRC --> URI[datasource://name URI + schema prompt-hint expansion]:::new
  KIND[MentionKind::Datasource]:::edit --> M1[code_match_rank arm]
  KIND --> M2[code_entry_path arm]
  KIND --> M3[empty_code_kind_rank arm]
  KIND --> M4[tier_rank arm]
  KIND --> M5[append_section_rows]
  classDef new fill:#dcfce7,stroke:#16a34a,color:#14532d;
  classDef edit fill:#fef9c3,stroke:#ca8a04,color:#713f12;
```

## 11. Build-order DAG (for the plan)

> **Event-driven infra (T0e) is foundational** and reusable (journey audit **U-4** is its second consumer). It lands before the TUI consumes datasource changes. T0 (DuckDB dep), catalog persistence (T2b), and idempotent setup-cell (T3) remain as grounded by codex `e38ffef2`.

```mermaid
flowchart LR
  T0[T0 DuckDB dep decision: feature-gate jute + accept compile cost]:::d --> T2
  T0e[T0e INFRA: daemon event_tx broadcast + subscribe server branch + orchestrator bridge + SpurEventBody::DatasourcesChanged]:::infra --> T6
  T1[T1 wire: AttachDatasource cmd + DatasourceKind + ts-rs]:::n --> T2[T2 catalog + duckdb introspection]:::n
  T2 --> T2b[T2b catalog persist/hydrate via notebook metadata + schema_version]:::n
  T2b --> T2e[T2e catalog emits DaemonEvent on attach/detach]:::infra
  T2e --> T0e
  T2b --> T3[T3 idempotent setup-cell generator + router arm]:::n
  T3 --> T7[T7 GUI DatasourceSidebar + control.ts]:::n
  T3 --> T6
  T2b --> T4[T4 notebook_list_datasources MCP tool]:::n
  T6[T6 TUI: slash add + DatasourcesChanged handler + MentionKind::Datasource + datasource:// URI]:::n --> T8[T8 E2E: drag CSV -> ask brain -> chart]:::n
  T4 --> T8
  T7 --> T8
  classDef n fill:#eef2ff,stroke:#6366f1,color:#3730a3;
  classDef d fill:#fee2e2,stroke:#dc2626,color:#7f1d1d;
  classDef infra fill:#fae8ff,stroke:#a21caf,color:#701a75;
```

## 12. Testing matrix

| Layer | Test | Asserts |
|---|---|---|
| jute (Rust) | `attach_introspects_csv_schema` | columns+row_count from a CSV fixture |
| jute (Rust) | `attach_inserts_idempotent_setup_cell` | one CREATE VIEW cell; re-attach no dup |
| spur-notebook | `router_handles_attach_datasource` | catalog upsert + DaemonEvent emitted |
| spur-notebook | `catalog_change_pushes_subscriber` | subscribe frame open → push frame on attach |
| spur-core | `bridge_reemits_as_spur_event` | daemon push → `SpurEventBody::DatasourcesChanged` on bus |
| spur-notebook | `list_datasources_returns_catalog` | MCP tool shape |
| spur-notebook | `catalog_hydrates_from_metadata` | reopen notebook → entries restored |
| spur-tui | `slash_notebook_data_add_parses` | path/name/group parse |
| spur-tui | `datasource_mention_section_renders` | `@` Data section + snapshot update |
| frontend (vitest) | `sidebar_attach_emits_command` | drag-drop → daemonControl |
| E2E | canonical demo | CSV → brain → chart renders |

## 13. Non-goals (MVP) / risks

**Non-goals:** remote DBs / object storage (scope C/D); credential/secret mgmt; data write-back/editing; brain-initiated attach; cross-notebook catalogs.

**Risks:** (1) `MentionKind` exhaustive fan-out — isolate as T6. (2) **new** daemon-side DuckDB dep in `jute` — *not* free reuse; gated decision T0 (compile-cost). (3) **event-channel infra (T0e)** is new cross-process plumbing (daemon `event_tx` → orchestrator subscribe loop → SpurEvent bus); reconnect/late-join handled by snapshot-on-subscribe + the §10 delta-gap pattern; shared with journey-audit **U-4**. (4) setup-cell idempotency on re-attach / rename — stable cell marker in metadata + source sentinel, update via `WriteCell`/`ApplyEdit`. (5) catalog persistence path normalization across machines.

*Next: → writing-plans over the §11 DAG → submit_plan to workers.*

## 14. Review refinements (codex `d1b22682` review)

Folded in regardless of the engine-placement decision:

- **Catalog persistence / hydration** — datasource metadata persists in notebook (or setup-cell) metadata and **hydrates the daemon catalog on open**; otherwise `@`-mentions + `notebook_list_datasources` vanish after restart. *(updates §5/§6)*
- **SQL escaping contract** — datasource names → identifier quoting; file paths → SQL literal escaping in the generated setup cell (injection / portability). *(updates §6)*
- **`row_count` cost** — `columns` is blocking at attach; `row_count` is best-effort, capped / cached / async on large CSV/JSON. *(updates §6/§9)*
- **`duckdb` pip availability** — attach must not require it; analysis cells need a clear failure mode + a setup-cell import check. *(updates §13)*
- **C/D extension + credential duplication** — remote/extension datasources would duplicate extension-loading + credentials across daemon and kernel; MVP stays **local-only** with a **re-review gate** before Postgres/S3/etc. *(updates §13 non-goals)*

## 15. Feasibility grounding (codex `e38ffef2`) — conditional go

Read-only grounding review against the live codebase (file:line anchors). **Verdict: conditional go** — write the plan, but with the corrections already folded into §5–§13.

| # | Component | Verdict | Anchor |
|---|-----------|---------|--------|
| 6 | Daemon→TUI `DatasourcesChanged` SpurEvent | **🔴 direct → bridged push (§16)** | daemon has no *direct* `Orchestrator.event_tx`; resolved by daemon `event_tx` + orchestrator bridge onto the existing bus |
| 7 | DuckDB as shared dep | NEEDS-DESIGN → T0 | version in workspace deps, but consumers feature-gated/CLI; notebook has no DuckDB dep |
| 1/6 | Representation-C catalog persistence | NEEDS-DESIGN → T2b | in-memory on `State` + notebook-metadata persist/hydrate |
| 6 | Setup-cell idempotent update | NEEDS-DESIGN → T3 | stable cell marker (metadata + sentinel), `WriteCell`/`ApplyEdit` |
| 10 | Datasource mentions → `ResourceLink` | NEEDS-DESIGN → T6 | define `datasource://` URI + schema prompt-hint |
| 7 | `/notebook data add` + mention grammar | NEEDS-DESIGN | `slash_commands.rs:9`, `registry.rs` exhaustive arms |
| 8 | Frontend sidebar/control | 🟢 FEASIBLE | `daemon/control.ts:16`, `HomePage.tsx:282` |
| 9 | Kernel + HTML/PNG viz path | 🟢 FEASIBLE | `run_cell.rs:46`, `notebook_store.rs:721`, `OutputView.tsx:74` |

**One real blocker, resolved event-driven (user directive):** codex's *direct*-event finding holds — a separate process can't write the in-process `tokio::broadcast`. But push is still achievable: the daemon gets its **own** `event_tx`, and the orchestrator (which owns the socket nonce) runs a persistent **subscribe loop** that re-emits daemon pushes as `SpurEventBody::DatasourcesChanged` onto the bus the TUI already consumes (`events.rs:745`, same path as `NotebookSocketReady`). Codex's polling option remains the documented fallback if the bridge proves costly. Everything else is "specify the contract," done above; viz + frontend-control paths are feasible as-is. See §16.

*Next: → writing-plans over the corrected §11 DAG → submit_plan to workers.*

## 16. Event-driven daemon→TUI channel (infra · T0e)

The push channel that makes datasource changes (and journey-audit **U-4**) proactive instead of polled. Reuses the daemon's **existing** Unix socket and the TUI's **existing** `SpurEvent` consumption — the only new pieces are an in-daemon broadcast and an orchestrator-side bridge.

```mermaid
sequenceDiagram
  autonumber
  participant ORCH as Orchestrator (spur-core)
  participant SOCK as daemon socket (notebook.v1)
  participant D as daemon (NotebookStore/Catalog + event_tx)
  participant BUS as SpurEvent broadcast (orchestrator.rs:227)
  participant TUI as spur-tui app loop (events.rs:745)

  Note over ORCH,SOCK: on register_notebook_socket (orchestrator.rs:387)
  ORCH->>SOCK: connect + "subscribe" frame (persistent, non-closing)
  SOCK->>D: hold connection; subscribe to event_tx
  D-->>ORCH: snapshot frame (current catalog) -- late-join safe
  loop on any catalog mutation
    D-->>ORCH: DaemonEvent::DatasourcesChanged push frame
    ORCH->>BUS: emit SpurEventBody::DatasourcesChanged{session, entries}
    BUS-->>TUI: handle_spur_event -> set_datasource_snapshot
  end
  Note over ORCH,SOCK: on disconnect: reconnect w/ backoff (mirrors notebook_daemon.rs retry)
```

**New infra, by anchor:**
- **Daemon broadcast** — add `event_tx: broadcast::Sender<DaemonEvent>` to the catalog/store; `attach`/`detach` send on it. (`spur-notebook` store)
- **Subscribe server branch** — in `handle_daemon_connection` (`spur-notebook/src/mcp/mod.rs:1354`), a new `"daemon":"notebook.v1"` sub-kind `"subscribe"` that, instead of one-shot `return` (line ~1382), writes a snapshot then streams `event_tx` frames until the peer drops.
- **Orchestrator bridge** — a background task spawned from `register_notebook_socket` (`orchestrator.rs:387`): connect to `control_socket_path(nonce)`, send `subscribe`, read frames, `self.emit(SpurEvent::now(SpurEventBody::DatasourcesChanged{..}))` per frame; reconnect with backoff. Reuses the framing in `spur-tui/src/notebook_daemon.rs:98`.
- **New event variant** — `SpurEventBody::DatasourcesChanged { session, entries }` in `spur-acp/src/domain/events.rs` (beside `NotebookSocketReady:531`).
- **TUI handler** — one arm in `events.rs` (beside `:397`) → `set_datasource_snapshot`.

**Why bridge through the orchestrator (not TUI-direct):** keeps a single event model — the TUI consumes exactly one stream (`SpurEvent`), as it does today; U-4 and any future daemon→TUI signal ride the same rail rather than each adding a bespoke socket reader.

*Next: → writing-plans over the corrected §11 DAG (T0e first) → submit_plan to workers.*